# Laboratorio 1 · ¿Qué sabe realmente esta base sobre nosotros?

**Explorar → Analizar → Experimentar → Decidir → Justificar → Reflexionar**

Leyes, Ética y Protección de Datos · Especialización en Análisis Estadístico para Ciencia de Datos · Docente: Wilson Sandoval Rodríguez

---

> **Versión del estudiante.** Las celdas están sin ejecutar y hay bloques marcados con `TODO` que usted debe completar. Ejecute de arriba hacia abajo y responda cada pregunta antes de continuar.

> Todos los datos son **sintéticos**. Ninguna persona real está representada.


## Qué va a hacer aquí

| | |
|:--|:--|
| **Aprenderá a** | Clasificar variables por el riesgo que introducen y medir el riesgo de reidentificación de una base |
| **Duración** | 35 minutos |
| **Evidencia** | Sus respuestas a las 9 preguntas y la base minimizada que construya |
| **Unidad** | 1 · Fundamentos legales |

Este cuaderno **no es una clase de programación**. El código es corto a
propósito. Lo que se evalúa es la decisión que usted toma después de leer la
salida.

---

# 1 · EXPLORAR

*¿Qué hay en esta base?*

In [ ]:
from pathlib import Path
import pandas as pd

# Funciona en tres sitios sin cambiar nada:
#   - dentro del repositorio (labs/ o raíz)
#   - en Google Colab
#   - en cualquier equipo con internet
RUTA = Path("../data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = Path("data/clientes_sinteticos.csv")
if not RUTA.exists():
    RUTA = "https://raw.githubusercontent.com/wilsonsr/leyes-etica-proteccion-datos/main/data/clientes_sinteticos.csv"

print("Origen de los datos:", RUTA)

In [ ]:
df = pd.read_csv(RUTA)
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)

print(f"Registros: {len(df)}   Variables: {df.shape[1]}")
df.head(3)

::: {.rds-card .decide}
**Pregunta 1.** Mire las tres primeras filas. Sin contar todavía las columnas: ¿cuántas de las que alcanza a ver le permitirían, por sí solas, llamar por teléfono a esa persona?
:::

In [ ]:
for i, col in enumerate(df.columns, start=1):
    print(f"{i:>2}. {col}")

In [ ]:
resumen = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "no_nulos": df.notna().sum(),
    "distintos": df.nunique(),
})
resumen["% distintos"] = (resumen["distintos"] / len(df) * 100).round(1)
resumen.sort_values("% distintos", ascending=False).head(12)

::: {.rds-card .decide}
**Pregunta 2.** Las variables con un porcentaje de valores distintos cercano al 100 % son casi siempre identificadores. ¿Cuáles aparecen arriba y cuál de ellas *no* esperaba encontrar ahí?
:::

---

# 2 · ANALIZAR

*¿Qué significa lo que hay?*

### Los faltantes no siempre son un problema de imputación

La pregunta útil no es *cuántos* faltan, sino *por qué* faltan.

In [ ]:
faltantes = (
    df.isna().sum().loc[lambda s: s > 0]
      .sort_values(ascending=False).to_frame("faltantes")
)
faltantes["%"] = (faltantes["faltantes"] / len(df) * 100).round(1)
faltantes

In [ ]:
# ¿Los faltantes de fecha_autorizacion están repartidos al azar?
pd.crosstab(
    df["origen_dato"],
    df["fecha_autorizacion"].isna().map({True: "sin fecha", False: "con fecha"}),
)

::: {.rds-card .riesgo}
**Pregunta 3.** Los faltantes de `fecha_autorizacion` no están repartidos al azar: se concentran en dos orígenes. ¿Cuáles? ¿Y qué significa, en términos prácticos, no tener fecha de autorización para esos registros?

Esto no es un problema de imputación. Es un problema de **evidencia**.
:::

### Clasificar por riesgo, no por tipo

El tipo que infiere `pandas` no dice nada sobre el riesgo. `object` puede ser un
nombre propio o una categoría inofensiva.

| Etiqueta | Significa |
|:--|:--|
| `directo` | Identifica a la persona por sí solo |
| `indirecto` | Identifica vía dispositivo, cuenta o ubicación precisa |
| `cuasi` | Solo no identifica; combinado con otros, sí |
| `comportamiento` | Lo que la persona hizo |
| `inferido` | Lo que la empresa dedujo |
| `proxy_sensible` | No es sensible, pero permite inferir uno |
| `control` | Metadato del tratamiento (origen, autorización) |
| `objetivo` | La variable que queremos predecir |

In [ ]:
# TODO: complete la clasificación de las 35 variables.
# Están resueltas las cinco primeras como ejemplo.
# Discuta las dudosas antes de decidir: latitud, longitud,
# ingresos_mensuales y categoria_top admiten más de una respuesta.

clasificacion = {
    "cliente_id": "indirecto",
    "nombre": "directo",
    "email": "directo",
    "telefono": "directo",
    "documento": "directo",
    # ... complete el resto
}

faltan = set(df.columns) - set(clasificacion)
print(f"Le faltan {len(faltan)} variables por clasificar:")
print(sorted(faltan))

In [ ]:
mapa = pd.Series(clasificacion, name="categoria").rename_axis("variable")
mapa.value_counts().to_frame("n_variables")

::: {.rds-card .decide}
**Pregunta 4.** Esta clasificación es **discutible a propósito**. Elija dos variables que usted habría puesto en otra categoría y defienda el cambio.

Candidatas frecuentes: `latitud`/`longitud` (¿indirecto o cuasi?), `ingresos_mensuales` (¿cuasi o proxy sensible?), `categoria_top` (¿comportamiento o proxy sensible, cuando el valor es `salud_bienestar`?).
:::

---

# 3 · EXPERIMENTAR

*¿Qué pasa si…?*

### El experimento central del laboratorio

Ninguna de estas columnas identifica sola. Veamos qué ocurre al combinarlas.

In [ ]:
def riesgo_unicidad(datos, columnas):
    # Cuenta cuántos registros quedan SOLOS en su grupo (k = 1).
    grupos = datos.groupby(columnas, dropna=False, observed=True).size()
    unicos = int((grupos == 1).sum())
    return {
        "variables": " + ".join(columnas),
        "combinaciones": int(len(grupos)),
        "registros k=1": unicos,
        "% en riesgo": round(unicos / len(datos) * 100, 1),
        "k mínimo": int(grupos.min()),
    }


combinaciones = [
    ["ciudad"],
    ["ciudad", "sexo"],
    ["ciudad", "sexo", "edad"],
    ["ciudad", "sexo", "edad", "ocupacion"],
    ["ciudad", "barrio", "sexo", "edad"],
    ["ciudad", "barrio", "sexo", "edad", "ocupacion", "estrato"],
]

pd.DataFrame([riesgo_unicidad(df, c) for c in combinaciones])

::: {.rds-card .riesgo}
**Pregunta 5.** Con una sola variable el riesgo es cero. ¿A partir de cuántas variables la mayoría de las personas de esta base queda sola en su grupo?

Anote el número. Es el argumento que va a necesitar la próxima vez que alguien diga «ya le quitamos los nombres».
:::

### Ahora al revés: reducir el riesgo y medirlo

La anonimización es un **resultado que se mide**, no una operación que se
aplica. Apliquemos tres generalizaciones y volvamos a medir.

In [ ]:
caso = df[["edad", "sexo", "ciudad", "ocupacion"]].copy()
caso_v2 = caso.copy()

# (a) Edad en rangos de 5 años
caso_v2["edad"] = pd.cut(caso_v2["edad"], bins=range(15, 90, 5)).astype(str)

# TODO (b): agrupe `ocupacion` en 3 o 4 categorías amplias con un diccionario.
# TODO (c): agrupe `ciudad` en regiones.
# Después ejecute la comparación de abajo y vea cuánto bajó el riesgo.

pd.DataFrame([
    {"versión": "original", **riesgo_unicidad(caso, ["edad", "sexo", "ciudad", "ocupacion"])},
    {"versión": "generalizada", **riesgo_unicidad(caso_v2, ["edad", "sexo", "ciudad", "ocupacion"])},
])

::: {.rds-card .prueba}
**Pregunta 6.** El porcentaje en riesgo bajó. ¿Bajó lo suficiente?

No hay un umbral universal. Lo que sí hay es una obligación: **decir cuál es el umbral que se aceptó y por qué**. Escriba el suyo.

Y la contrapartida: ¿qué análisis dejó de ser posible con la base generalizada? ¿Vale la pena?
:::

---

# 4 · DECIDIR

*¿Qué hacemos?*

### Minimizar no es «quitar columnas»

Minimizar es responder, para cada columna, **por qué la necesito para esta
finalidad concreta**. La finalidad declarada aquí es: *predecir abandono para
una campaña de retención*.

In [ ]:
# TODO: construya la lista mínima de variables que justificaría ante la
# Dirección Comercial. Debe poder decir en una frase por qué cada una está.

variables_minimas = [
    "cliente_id",   # necesario para poder actuar sobre el cliente
    # ... complete
    "churn",        # variable objetivo
]

df_min = df[variables_minimas].copy()
print(f"Original: {df.shape[1]} variables  ->  Minimizada: {df_min.shape[1]}")

### Seudonimizar no es anonimizar

Reemplazamos el identificador por un hash. Es buena práctica **y no es
anonimización**: la celda siguiente muestra por qué.

In [ ]:
import hashlib

SAL = "curso-lepd-2026"   # en producción: secreto, rotado y fuera del código


def seudonimizar(valor, sal=SAL, largo=12):
    return hashlib.sha256(f"{sal}{valor}".encode("utf-8")).hexdigest()[:largo].upper()


df_seudo = df_min.copy()
df_seudo["cliente_id"] = df_seudo["cliente_id"].map(seudonimizar)
df_seudo.head(4)

In [ ]:
# La tabla de equivalencias: esto es lo que impide llamarlo anonimización.
tabla_equivalencias = pd.DataFrame({
    "cliente_id": df["cliente_id"],
    "seudonimo": df["cliente_id"].map(seudonimizar),
    "nombre": df["nombre"],
})
tabla_equivalencias.head(4)

::: {.rds-card .decision}
**Pregunta 7.** Mientras exista la tabla anterior, los datos siguen siendo datos personales.

Y la tabla **tiene que existir**, porque sin ella la empresa no puede contactar al cliente que el modelo señaló. Ese es el nudo: la finalidad del proyecto —actuar sobre personas concretas— es incompatible con la anonimización real.

¿Dónde debería vivir esa tabla y quién debería poder leerla?
:::

---

# 5 · JUSTIFICAR

*¿Con qué evidencia?*

Una decisión sin evidencia no cuenta como decisión. Cerramos escribiendo el
registro del laboratorio.

In [ ]:
# TODO: complete el registro con SUS decisiones y SU umbral.

registro = pd.DataFrame([{
    "proyecto": "DataMarket · modelo de abandono",
    "fecha": str(pd.Timestamp("today").date()),
    "variables_originales": df.shape[1],
    "variables_conservadas": df_min.shape[1],
    "criterio_minimizacion": "TODO: ¿por qué esas y no otras?",
    "umbral_riesgo_aceptado": "TODO: ¿qué k mínimo acepta y por qué?",
    "registros_sin_evidencia_origen": int(df["fecha_autorizacion"].isna().sum()),
    "decision_sobre_esos_registros": "TODO: ¿excluir, aislar, regularizar?",
    "responsable": "TODO: su nombre",
}])

registro.T.rename(columns={0: "valor"})

::: {.rds-card .decision}
**Pregunta 8.** Suponga que esta base se va a compartir con un proveedor externo de analítica. Con lo que midió hoy, ¿la entregaría?

Si su respuesta es «sí, con condiciones», enumere las condiciones.
:::

---

# 6 · REFLEXIONAR

*¿Y en mi trabajo?*

::: {.rds-card .reflexiona}
**Pregunta 9 · de salida.** Tome una base con la que trabaje realmente. Ejecute mentalmente el bloque 3 sobre ella: ¿cuántas variables cuasi-identificadoras tiene?

> **¿Qué tendría que cambiar en mi proyecto de datos?**

Tres líneas. Nombre una variable concreta y un cambio ejecutable.
:::

---

## Lo que hicimos

1. Exploramos el tamaño real del problema: 1 512 registros × 35 variables.
2. Descubrimos que los faltantes de `fecha_autorizacion` no eran ruido sino la
   huella del origen de los datos.
3. Clasificamos las variables por riesgo, no por tipo.
4. Medimos que cuatro cuasi-identificadores dejan sola a la gran mayoría de las
   personas de la base.
5. Construimos una base minimizada para una finalidad declarada.
6. Seudonimizamos y vimos por qué eso no anonimiza.
7. Generalizamos y **medimos** cuánto bajó el riesgo, y cuánta utilidad costó.

**Siguiente paso:** Laboratorio 2 · auditoría de un pipeline completo, y el
dilema entre un modelo que predice mejor y uno que usa menos.

**Marco legal relacionado:** la excepción para fines estadísticos y científicos
del art. 10 de la Ley 1581 de 2012 exige suprimir la identidad de los titulares.
Este laboratorio muestra por qué eso no es trivial.